# Boost — 2차 부스터 (stage2 결합)

세 전문가의 **로짓만** 받아 3-class 확률을 내는 결합기. 두 구성을 같은 조건으로 비교한다.

| arm | 입력 | 열 |
|---|---|---|
| **AT**  | ABE 로짓 + TB 로짓 | 3 + 2 = 5 |
| **ATR** | ABE 로짓 + TB 로짓 + Arith 로짓 | 3 + 2 + 9 = 14 |

각 arm 을 부스터 3종(LGBM·XGBoost·CatBoost)으로 학습 → 6개 구성.

**검증** — stage2 10,748(valid 5,000 + holdout 5,748)로 5-fold 교차검증.
fold 마다 20%를 평가로 떼고, 남은 80%에서 다시 **10%(전체 기준)를 ES 전용**으로 뗀다.
지표는 **5 fold 값의 평균 ± 표준편차**다 — 5개 모델의 예측을 평균한 앙상블 점수가 아니다.
(앙상블 점수는 fold 모델이 서로 다른 데이터를 본 이득까지 섞여 배포 성능을 과대평가한다.)

**배포** — CV 는 측정 전용이고, 배포용 모델은 마지막 셀에서 stage2 **전량**으로 다시 적합해 저장한다.

**로짓 계약 (중요)** — 전문가마다 출력 형상과 swap 거동이 다르다. 이걸 틀리면 조용히 오염된다:

| 전문가 | 형상 | swap (A↔B 교환) 시 |
|---|---|---|
| ABE | (N, 3) `[z_a, z_b, z_tie]` | `z_a ↔ z_b` 교환 |
| **TB** | **(N, 2)** `[z_notie, z_tie]` | **불변 — 교환·재정렬 금지** |
| Arith | (N, 9) 부스터 3종 × `[p_a, p_b, p_tie]` (log p) | 블록마다 앞 두 열 교환 |

TB 는 2-class 라 `[1,0,2]` 재정렬을 적용하면 죽거나(운이 좋으면) 값이 뒤섞인다.
또한 TB 는 빈 응답 행에서 구간 마스크가 전부 0 이 되는데, 배포본 `AttnPool` 이
`torch.nan_to_num(w)` 로 0 벡터를 만들어 유한값을 보장한다(`Boost/Train_TB_run.ipynb` cell 25).
이 노트북은 그 결과만 읽으므로 **로드 직후 NaN·형상·id 정렬을 전부 assert** 해 계약을 강제한다.

### 설정 / Setup

In [ ]:
import os, json, glob, time, shutil
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
np.random.seed(SEED)

# 경로는 절대경로로 고정 — 노트북 CWD 에 따라 산출물이 새는 사고를 막는다
# absolute paths: the notebook CWD must not decide where artifacts come from / go
LLM       = Path('.')
ABE_ROOT  = LLM / 'V5/aws/outputs/best_checkpoints'
TB_ROOT   = LLM / 'V6/pod/outputs/best_checkpoints'
ARITH_NPZ = LLM / 'Arith/outputs/arith_experts.npz'
ORIGIN    = LLM / 'Arith/origin'          # V5/origin 과 동일한 동결 분할 (id 대조로 검증)
OUT       = LLM / 'Boost'
MODEL_DIR = OUT / 'model'

# ── 손잡이 ──
CKPT_SELECT = 'valid'   # 'valid': 프로토콜 준수(선택은 valid, hold 무접촉) | 'hold': hold 기준
ARITH_MODE  = 'all'     # 'all': 부스터 3종 로짓 9열 그대로 | 'mean': logp 평균 3열
USE_SWAP    = True      # 학습 swap 증강 + 예측 swap-TTA (전 단계와 동일 규율)
N_SPLITS    = 5
INNER_FRAC  = 0.10      # ES 전용 비율 — stage2 전체 기준 10%

for p in (ABE_ROOT, TB_ROOT, ARITH_NPZ, ORIGIN / 'valid.csv', ORIGIN / 'holdout.csv'):
    assert p.exists(), f'경로 없음: {p}'
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'ckpt 선택 {CKPT_SELECT} | arith {ARITH_MODE} | swap {USE_SWAP} | {N_SPLITS}-fold, ES {INNER_FRAC:.0%}')

### stage2 로드 — valid 5,000 + holdout 5,748 (이 순서가 전 산출물의 행 순서다)

In [ ]:
def labels_of(d):
    y = np.select([d['winner_model_a'] == 1, d['winner_model_b'] == 1, d['winner_tie'] == 1],
                  [0, 1, 2], default=-1).astype(np.int64)
    assert (y != -1).all(), 'invalid labels!'
    return y

va = pd.read_csv(ORIGIN / 'valid.csv')
ho = pd.read_csv(ORIGIN / 'holdout.csv')
assert len(va) == 5000 and len(ho) == 5748, f'분할 크기 {len(va)}/{len(ho)}'
IDS = np.concatenate([va['id'].to_numpy(), ho['id'].to_numpy()])
Y = np.concatenate([labels_of(va), labels_of(ho)])
HALF = len(va)                                   # 앞 5,000 = valid, 뒤 5,748 = holdout
assert len(np.unique(IDS)) == len(IDS), 'id 중복'
prior = np.bincount(Y, minlength=3) / len(Y)
print(f'stage2 {len(IDS):,}행 (valid {HALF:,} + hold {len(IDS) - HALF:,}) | '
      f'사전분포 a/b/tie {np.round(prior, 4)}')

### 전문가 체크포인트 선정 + 로짓 로드

각 체크포인트의 `logits_export.npz` 에서 valid TTA 로짓으로 T 를 피팅하고 valid/hold 를 함께 채점한다
(각 학습 노트북의 검증 셀과 같은 공식). **선택은 `CKPT_SELECT` 기준 한쪽으로만** 한다 —
hold 로 고르면 그 hold 가 곧 부스터 평가의 절반이라 낙관이 섞인다. 기본값 `valid` 가 §1.6 프로토콜이다.

T 는 여기서 **적용하지 않는다.** 트리는 한 열을 상수로 나눠도 분할 순서가 안 바뀌어 결과가 동일하고,
부스터가 직접 확률을 내므로 별도 보정이 필요 없다. T 는 선택 지표를 만드는 데만 쓴다.

In [ ]:
import torch
import torch.nn as nn

def fit_temperature(logits, labels):
    z = torch.tensor(logits, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)
    logT = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = nn.functional.cross_entropy(z / logT.exp(), y)
        loss.backward()
        return loss
    opt.step(closure)
    return float(logT.exp().detach())

# valid 로 T 피팅 -> valid/hold 각각 채점. ABE = 3-class ll, TB = binary tie ll.
def score_ckpt(tag, npz_path):
    z = np.load(npz_path)
    v = (z['valid_fwd'] + z['valid_swp']) / 2
    h = (z['hold_fwd'] + z['hold_swp']) / 2
    vl, hl = z['valid_labels'], z['hold_labels']
    if tag == 'abe':
        T = fit_temperature(v, vl)
        sm = lambda x: torch.softmax(torch.tensor(x / T, dtype=torch.float32), 1).numpy()
        return log_loss(vl, sm(v), labels=[0, 1, 2]), log_loss(hl, sm(h), labels=[0, 1, 2]), T
    T = fit_temperature(v, (vl == 2).astype(np.int64))
    sm = lambda x: torch.softmax(torch.tensor(x / T, dtype=torch.float32), 1).numpy()[:, 1]
    b = lambda p, l: log_loss((l == 2).astype(int), np.c_[1 - p, p], labels=[0, 1])
    return b(sm(v), vl), b(sm(h), hl), T

def pick_and_load(tag, root, n_class):
    rows = []
    for p in sorted(glob.glob(str(root / 'checkpoint-*')), key=lambda s: int(s.rsplit('-', 1)[1])):
        lv, lh, T = score_ckpt(tag, Path(p) / 'logits_export.npz')
        rows.append((int(p.rsplit('-', 1)[1]), lv, lh, T, p))
    key = 1 if CKPT_SELECT == 'valid' else 2
    best = min(rows, key=lambda r: r[key])
    metric = '3-class ll' if tag == 'abe' else 'binary tie ll'
    print(f'[{tag.upper()}] {metric:<14} {"step":>6} {"valid":>8} {"hold":>8} {"T":>7}')
    for s, lv, lh, T, _ in rows:
        mark = '  <-- 선택' if s == best[0] else ''
        print(f'{"":>21} {s:>6} {lv:8.4f} {lh:8.4f} {T:7.4f}{mark}')

    z = np.load(Path(best[4]) / 'logits_export.npz')
    L = np.vstack([(z['valid_fwd'] + z['valid_swp']) / 2,
                   (z['hold_fwd'] + z['hold_swp']) / 2]).astype(np.float64)
    zid = np.concatenate([z['valid_ids'], z['hold_ids']])
    zlab = np.concatenate([z['valid_labels'], z['hold_labels']])
    # ── 계약 강제 ── 형상·정렬·유한성. 하나라도 어긋나면 조용한 오염이 된다
    assert L.shape == (len(IDS), n_class), f'{tag} 형상 {L.shape} != {(len(IDS), n_class)}'
    assert (zid == IDS).all(), f'{tag} id 정렬 불일치 — 행이 어긋난 채 학습된다'
    assert (zlab == Y).all(), f'{tag} 라벨 불일치'
    assert np.isfinite(L).all(), f'{tag} 비유한값 {int((~np.isfinite(L)).sum())}개 (빈 응답 가드 확인)'
    return best[0], best[3], L

ABE_STEP, ABE_T, L_ABE = pick_and_load('abe', ABE_ROOT, 3)
print()
TB_STEP, TB_T, L_TB = pick_and_load('tb', TB_ROOT, 2)
assert L_TB.shape[1] == 2, 'TB 는 2-class 다 — [1,0,2] 재정렬 금지'

### Arith 로짓 로드 — 부스터 3종의 log p

In [ ]:
za = np.load(ARITH_NPZ)
aid = np.concatenate([za['ids_valid'], za['ids_hold']])
alab = np.concatenate([za['y_valid'], za['y_hold']])
assert (aid == IDS).all(), 'Arith id 정렬 불일치'
assert (alab == Y).all(), 'Arith 라벨 불일치'

ARITH_MODELS = ('lgbm', 'xgb', 'cat')
blocks = [np.vstack([za[f'logp_valid__{m}'], za[f'logp_hold__{m}']]).astype(np.float64)
          for m in ARITH_MODELS]
if ARITH_MODE == 'mean':
    L_ARI = np.mean(blocks, axis=0)                       # 3열 — logp 평균
    ARI_COLS = [f'ari_mean_{c}' for c in ('a', 'b', 'tie')]
else:
    L_ARI = np.hstack(blocks)                             # 9열 — 부스터별로 다 준다
    ARI_COLS = [f'ari_{m}_{c}' for m in ARITH_MODELS for c in ('a', 'b', 'tie')]
assert np.isfinite(L_ARI).all(), 'Arith 비유한값'
print(f'Arith {L_ARI.shape} ({ARITH_MODE}) | 열: {ARI_COLS}')

### 행렬 조립 + swap 매핑

swap 은 **열 치환 하나**로 표현한다 — 순열 배열이면 `perm[perm] == 항등` 을 직접 검사할 수 있어
부호반전·역수처럼 손으로 짠 변환보다 실수 여지가 없다. TB 두 열은 순열에서 제자리에 남는다.

In [ ]:
ABE_COLS = ['abe_z_a', 'abe_z_b', 'abe_z_tie']
TB_COLS = ['tb_z_notie', 'tb_z_tie']

X_AT = np.hstack([L_ABE, L_TB])                   # arm AT
X_ATR = np.hstack([L_ABE, L_TB, L_ARI])           # arm ATR
COLS_AT, COLS_ATR = ABE_COLS + TB_COLS, ABE_COLS + TB_COLS + ARI_COLS

# A<->B 교환 시의 열 순열. ABE/Arith 는 a<->b 교환, TB 는 불변(제자리).
def make_perm(cols):
    idx = {c: i for i, c in enumerate(cols)}
    perm = np.arange(len(cols))
    pairs = [('abe_z_a', 'abe_z_b')]
    if ARITH_MODE == 'mean':
        pairs.append(('ari_mean_a', 'ari_mean_b'))
    else:
        pairs += [(f'ari_{m}_a', f'ari_{m}_b') for m in ARITH_MODELS]
    for a, b in pairs:
        if a in idx and b in idx:
            perm[idx[a]], perm[idx[b]] = idx[b], idx[a]
    return perm

PERM = {'AT': make_perm(COLS_AT), 'ATR': make_perm(COLS_ATR)}
for k, p in PERM.items():
    cols = COLS_AT if k == 'AT' else COLS_ATR
    assert (p[p] == np.arange(len(p))).all(), f'{k} 순열이 involution 이 아니다'
    fixed = [c for i, c in enumerate(cols) if p[i] == i]
    assert all(c.startswith('tb_') or c.endswith('_tie') for c in fixed), f'{k} 고정열 이상: {fixed}'
print('swap 순열 involution 통과 | 고정열(=swap 불변):',
      [c for i, c in enumerate(COLS_ATR) if PERM['ATR'][i] == i])

def swap_y(y):
    return np.where(y == 0, 1, np.where(y == 1, 0, 2)).astype(y.dtype)

ARMS = {'AT': (X_AT, COLS_AT), 'ATR': (X_ATR, COLS_ATR)}
for k, (X, c) in ARMS.items():
    print(f'  arm {k:<4} {X.shape}  {c}')

### 지표 — 축 분해 (총 ll = tie축 + 승부축)

임의의 3-class 확률벡터에 성립하는 항등식이라 매 fold 마다 검산까지 한다.
게이트가 tie 축만 갈아끼우는 구조이므로, 부스터가 어느 축에서 벌었는지 보는 것이 핵심이다.

In [ ]:
EPS = 1e-15

def per_row_ll(p, y):
    p = np.clip(np.asarray(p, np.float64), EPS, 1.0)
    p = p / p.sum(1, keepdims=True)
    return -np.log(p[np.arange(len(y)), y])

def axis_report(p, y):
    p = np.clip(np.asarray(p, np.float64), EPS, 1.0)
    p = p / p.sum(1, keepdims=True)
    total = per_row_ll(p, y).mean()
    it = y == 2
    t = np.clip(p[:, 2], EPS, 1 - EPS)
    tie_axis = float(-np.where(it, np.log(t), np.log(1 - t)).mean())
    nz = ~it
    rA = np.clip(p[nz, 0] / (p[nz, 0] + p[nz, 1]), EPS, 1 - EPS)
    win_axis = float(-np.where(y[nz] == 0, np.log(rA), np.log(1 - rA)).sum() / len(y))
    assert abs(total - tie_axis - win_axis) < 1e-6, '축 분해 검산 실패'
    return {'총 ll': float(total), 'tie축': tie_axis, '승부축': win_axis,
            'tie AUC': float(roc_auc_score(it.astype(int), t)),
            '승부 AUC': float(roc_auc_score((y[nz] == 0).astype(int), rA)),
            'acc%': float((p.argmax(1) == y).mean() * 100)}

# Δ = mean(ll_a − ll_b) 의 95% CI. 양수면 b 가 좋다.
def paired_ci(ll_a, ll_b, n_boot=2000, seed=SEED):
    d = np.asarray(ll_a) - np.asarray(ll_b)
    rng = np.random.default_rng(seed)
    bs = d[rng.integers(0, len(d), (n_boot, len(d)))].mean(1)
    return float(d.mean()), float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))

def fmt_ci(m, lo, hi, thresh=0.0005):
    sig = '유의' if (lo > 0 or hi < 0) else '노이즈'
    return f'{m:+.4f} [{lo:+.4f},{hi:+.4f}] {sig}' + (' ★' if sig == '유의' and abs(m) >= thresh else '')

### 부스터 3종 + 5-fold 하네스

파라미터는 Arith/Train.ipynb 와 같은 계열이되 **입력이 5~14열뿐이므로 용량을 낮췄다**
(leaves 63 → 15, depth 8 → 4). 10,748행에 5열짜리 문제에 큰 트리를 쓰면 ES 만으로는 과적합을 못 막는다.

In [ ]:
import inspect
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

def make_lgbm():
    return lgb.LGBMClassifier(objective='multiclass', num_class=3, learning_rate=0.03,
                              n_estimators=3000, num_leaves=15, min_child_samples=60,
                              colsample_bytree=0.9, subsample=0.9, subsample_freq=1,
                              reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbosity=-1)

def make_xgb():
    return xgb.XGBClassifier(objective='multi:softprob', num_class=3, learning_rate=0.03,
                             n_estimators=3000, max_depth=4, min_child_weight=20,
                             subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
                             tree_method='hist', eval_metric='mlogloss',
                             early_stopping_rounds=100, random_state=SEED, n_jobs=-1, verbosity=0)

def make_cat():
    return cb.CatBoostClassifier(loss_function='MultiClass', eval_metric='MultiClass',
                                 iterations=3000, learning_rate=0.03, depth=4,
                                 l2_leaf_reg=6.0, early_stopping_rounds=100,
                                 bootstrap_type='Bernoulli', subsample=0.9, border_count=254,
                                 random_seed=SEED, verbose=0, allow_writing_files=False)

FACTORY = {'lgbm': make_lgbm, 'xgb': make_xgb, 'cat': make_cat}
_LGB_NEW = 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters

def fit_one(name, m, Xf, yf, X_es, y_es):
    if name == 'lgbm':
        ekw = {'eval_X': X_es, 'eval_y': y_es} if _LGB_NEW else {'eval_set': [(X_es, y_es)]}
        m.fit(Xf, yf, **ekw, callbacks=[lgb.early_stopping(100, verbose=False)])
    elif name == 'xgb':
        m.fit(Xf, yf, eval_set=[(X_es, y_es)], verbose=False)
    else:
        m.fit(Xf, yf, eval_set=(X_es, y_es), use_best_model=True)
    return m

def best_iter(name, m):
    if name == 'lgbm':
        return int(m.booster_.best_iteration or 0)
    if name == 'xgb':
        return int(getattr(m, 'best_iteration', 0) or 0)
    return int(m.get_best_iteration() or 0)

# swap-TTA — 교환 뷰의 예측은 클래스도 되돌려야 한다([1,0,2]). TB 열은 순열이 제자리라 안전.
def predict_tta(m, Xp, perm):
    p = m.predict_proba(Xp)
    if not USE_SWAP:
        return p
    return (p + m.predict_proba(Xp[:, perm])[:, [1, 0, 2]]) / 2

# fold 마다 평가 20% / ES 10%(전체 기준) / 나머지 학습. 지표는 fold 값 자체를 모은다.
def cv_boost(name, X, y, perm, n_splits=N_SPLITS, inner_frac=INNER_FRAC, seed=SEED):
    n_inner = int(round(inner_frac * len(y)))
    skf = StratifiedKFold(n_splits, shuffle=True, random_state=seed)
    oof = np.zeros((len(y), 3))
    rows, iters = [], []
    for k, (rest, te) in enumerate(skf.split(X, y)):
        tr, es = train_test_split(rest, test_size=n_inner, stratify=y[rest],
                                  random_state=seed + k, shuffle=True)
        if USE_SWAP:
            Xf = np.vstack([X[tr], X[tr][:, perm]])
            yf = np.hstack([y[tr], swap_y(y[tr])])
        else:
            Xf, yf = X[tr], y[tr]
        m = fit_one(name, FACTORY[name](), Xf, yf, X[es], y[es])
        p = predict_tta(m, X[te], perm)
        oof[te] = p
        rows.append(axis_report(p, y[te]))
        iters.append(best_iter(name, m))
    fold_df = pd.DataFrame(rows)
    return {'fold': fold_df, 'mean': fold_df.mean(), 'std': fold_df.std(ddof=1),
            'oof': oof, 'iters': iters}

### 실행 — 2 arm × 부스터 3종

In [ ]:
R = {}
for arm, (X, cols) in ARMS.items():
    for name in FACTORY:
        t0 = time.time()
        R[(arm, name)] = cv_boost(name, X, Y, PERM[arm])
        r = R[(arm, name)]
        print(f"{arm:<4} {name:<5} 총 ll {r['mean']['총 ll']:.4f} ± {r['std']['총 ll']:.4f}"
              f"   tie축 {r['mean']['tie축']:.4f}  승부축 {r['mean']['승부축']:.4f}"
              f"   iters {r['iters']}  ({time.time() - t0:.0f}s)")

### 결과 — fold 평균이 판정 지표

`blend-w11` 0.9844 / `w11+게이트(전부)` 0.9812 는 V4 가 같은 stage2 10,748 에서 낸 값이라 직접 비교된다.
단 그쪽은 5-fold OOF 전체 점수이고 여기는 fold 평균이라, 0.001 급 차이는 프로토콜 차이로도 난다.

In [ ]:
V4_BLEND, V4_GATE = 0.9844, 0.9812      # V4 test10 (같은 stage2 10,748)

tbl = pd.DataFrame({f'{a}·{n}': R[(a, n)]['mean'] for a, n in R}).T
tbl['ll std'] = [R[(a, n)]['std']['총 ll'] for a, n in R]
tbl['vs V4게이트'] = tbl['총 ll'] - V4_GATE
display(tbl.sort_values('총 ll').round(4))

print('\n[fold 별 총 ll]')
fold_tbl = pd.DataFrame({f'{a}·{n}': R[(a, n)]['fold']['총 ll'].values for a, n in R},
                        index=[f'fold{k}' for k in range(N_SPLITS)])
display(fold_tbl.round(4))

print('\n[기준선]', f'사전분포 {-np.log(prior[Y]).mean():.4f}', f'| 균등 {np.log(3):.4f}',
      f'| V4 blend {V4_BLEND} | V4 게이트 {V4_GATE}')

### arm 비교 — Arith 로짓을 넣으면 실제로 좋아지나

같은 fold·같은 행에서 나온 OOF 예측이라 paired bootstrap 이 성립한다.
이게 "산술 전문가가 앙상블에 기여하는가"에 대한 **배포 충실 판정**이다 —
모든 로짓이 stage2 에서 out-of-sample 이고 구조도 실제 파이프라인과 같다.

In [ ]:
print('=== AT -> ATR (양수 = Arith 로짓 추가가 이득, 문턱 0.0005) ===')
for name in FACTORY:
    a, b = R[('AT', name)]['oof'], R[('ATR', name)]['oof']
    print(f'  {name:<5} ' + fmt_ci(*paired_ci(per_row_ll(a, Y), per_row_ll(b, Y))))

print('\n=== 부스터 간 (arm 안에서, lgbm 기준) ===')
for arm in ARMS:
    base = per_row_ll(R[(arm, 'lgbm')]['oof'], Y)
    for name in ('xgb', 'cat'):
        print(f'  {arm:<4} lgbm -> {name:<5} '
              + fmt_ci(*paired_ci(base, per_row_ll(R[(arm, name)]['oof'], Y))))

print('\n=== 3종 logp 평균 앙상블 (arm 별) ===')
for arm in ARMS:
    lp = np.mean([np.log(np.clip(R[(arm, n)]['oof'], EPS, 1)) for n in FACTORY], axis=0)
    p_ens = np.exp(lp)
    p_ens /= p_ens.sum(1, keepdims=True)
    rep = axis_report(p_ens, Y)
    best_single = min(FACTORY, key=lambda n: log_loss(Y, R[(arm, n)]['oof'], labels=[0, 1, 2]))
    d = paired_ci(per_row_ll(R[(arm, best_single)]['oof'], Y), per_row_ll(p_ens, Y))
    print(f"  {arm:<4} 앙상블 총 ll {rep['총 ll']:.4f} (tie축 {rep['tie축']:.4f} / "
          f"승부축 {rep['승부축']:.4f})  vs 단독최선({best_single}) " + fmt_ci(*d))

print('\n[valid/hold 반쪽 분리 — 이득이 한쪽에만 있으면 선택 편향 신호]')
for arm in ARMS:
    for name in FACTORY:
        o = R[(arm, name)]['oof']
        lv = log_loss(Y[:HALF], o[:HALF], labels=[0, 1, 2])
        lh = log_loss(Y[HALF:], o[HALF:], labels=[0, 1, 2])
        print(f'  {arm:<4} {name:<5} valid {lv:.4f}  hold {lh:.4f}  차 {lv - lh:+.4f}')

### T 스케일링을 부스터 출력에 걸어야 하나 (평가만, 미적용)

입력 로짓에 T 를 거는 건 무의미하다 — 트리는 한 열을 상수로 나눠도 분할 순서가 안 바뀐다.
논점은 **출력** 쪽이다: swap-TTA 로 확률 두 벌을 평균하면 중앙으로 당겨져 과소확신이 생길 수 있고,
T 는 그걸 되돌리는 스칼라 하나다.

여기서는 **적용하지 않고 얼마나 벌리는지만 잰다.** T 는 두 방식으로 구한다:
- **교차적합** — fold k 의 T 를 나머지 4 fold 의 OOF 로 피팅해 fold k 에 적용. 자기적합 낙관이 없다. (판정 기준)
- 자기적합 — OOF 전체로 피팅해 그대로 적용. 스칼라 1개라 낙관은 ~1e-4 지만 참고로만 본다.

교차적합 이득이 문턱 0.0005 를 넘으면 배포 파이프라인에 T 를 넣는다.

In [ ]:
def apply_T(p, T):
    z = np.log(np.clip(p, EPS, 1.0)) / T          # 확률 -> 로짓(상수항은 softmax 가 흡수)
    z -= z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

# cv_boost 와 동일한 분할을 재현 — StratifiedKFold 는 y 와 행 수만 보므로 더미 X 로 같은 fold 가 나온다
_skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
FOLD = np.zeros(len(Y), int)
for _k, (_, _te) in enumerate(_skf.split(np.zeros((len(Y), 1)), Y)):
    FOLD[_te] = _k
assert np.bincount(FOLD).min() > 0

OOF_T = {}
print(f'{"구성":<12} {"raw ll":>8} {"T(교차)":>9} {"Δ":>8}  {"T 값":<22} {"자기적합":>9}')
for arm, name in R:
    p = R[(arm, name)]['oof']
    pT = np.zeros_like(p)
    Ts = []
    for k in range(N_SPLITS):
        tr, te = FOLD != k, FOLD == k
        T = fit_temperature(np.log(np.clip(p[tr], EPS, 1.0)), Y[tr])   # 다른 fold 로만 피팅
        pT[te] = apply_T(p[te], T)
        Ts.append(T)
    T_self = fit_temperature(np.log(np.clip(p, EPS, 1.0)), Y)
    OOF_T[(arm, name)] = pT
    ll_raw = log_loss(Y, p, labels=[0, 1, 2])
    ll_cv = log_loss(Y, pT, labels=[0, 1, 2])
    ll_self = log_loss(Y, apply_T(p, T_self), labels=[0, 1, 2])
    print(f'{arm + "·" + name:<12} {ll_raw:8.4f} {ll_cv:9.4f} {ll_raw - ll_cv:+8.4f}  '
          f'{"[" + " ".join(f"{t:.2f}" for t in Ts) + "]":<22} {ll_self:9.4f}')

print('\n=== raw -> T(교차적합) paired bootstrap (양수 = T 가 이득, 문턱 0.0005) ===')
for arm, name in R:
    d = paired_ci(per_row_ll(R[(arm, name)]['oof'], Y), per_row_ll(OOF_T[(arm, name)], Y))
    print(f'  {arm + "·" + name:<12} ' + fmt_ci(*d))
print('\n  T > 1 이면 과신(=TTA 평균에도 불구하고 날카롭다), T < 1 이면 과소확신이라 더 날카롭게 해야 한다.')

### 손실 해부 — 어디서 얼마나 깎이나

총 ll 은 평균이라 "어느 행이 얼마를 물어냈는지"를 감춘다. 다섯 갈래로 쪼갠다:

1. **클래스별 기여** — A승/B승/tie 각각이 총 손실의 몇 %를 차지하나
2. **집중도** — 상위 몇 %의 어려운 행이 손실의 몇 %를 먹나 (꼬리가 지배하면 평균 개선은 그 꼬리 싸움이다)
3. **캘리브레이션** — 신뢰도 구간별 예측 확신 vs 실제 정답률, ECE (위 T 판단의 근거)
4. **축 + 회수율** — tie축/승부축이 각각 무정보 기준 대비 몇 %를 회수했나. 오라클(축 하나를 완벽히
   맞혔을 때 남는 손실)이 곧 다른 축의 값이므로 남은 여지가 어디인지 바로 읽힌다
5. **전문가 합의 층화** — ABE 와 TB 의 tie 판정이 엇갈리는 행에서 부스터가 중재를 해내고 있나

In [ ]:
NOINFO_TIE = float(-(prior[2] * np.log(prior[2]) + (1 - prior[2]) * np.log(1 - prior[2])))
NOINFO_WIN = float(np.log(2) * (1 - prior[2]))          # 승부축은 비-tie 비율로 스케일된다

# 전문가 자체 판정 — 층화용 (부스터 입력 원본에서 직접 뽑는다)
_abe_p = np.exp(L_ABE - L_ABE.max(1, keepdims=True))
_abe_p /= _abe_p.sum(1, keepdims=True)
_tb_p = np.exp(L_TB - L_TB.max(1, keepdims=True))
_tb_p /= _tb_p.sum(1, keepdims=True)
ABE_TIE = _abe_p[:, 2] > 0.5          # ABE 가 tie 라고 보나
TB_TIE = _tb_p[:, 1] > 0.5            # TB 가 tie 라고 보나 (2-class 의 1번 열)

def dissect(p, y, tag):
    e = per_row_ll(p, y)
    tot = e.mean()
    rep = axis_report(p, y)
    print(f'\n{"=" * 78}\n[{tag}]  총 ll {tot:.4f}')

    print('\n  1) 클래스별 손실 기여')
    print(f'     {"클래스":<6} {"n":>6} {"비율":>7} {"평균손실":>9} {"기여":>8} {"기여%":>7}')
    for c, nm in enumerate(('A승', 'B승', 'tie')):
        m = y == c
        print(f'     {nm:<6} {int(m.sum()):>6,} {m.mean():7.3f} {e[m].mean():9.4f} '
              f'{e[m].sum() / len(y):8.4f} {e[m].sum() / e.sum() * 100:6.1f}%')

    print('\n  2) 손실 집중도 (어려운 행 순)')
    o = np.argsort(-e)
    for q in (0.05, 0.10, 0.20, 0.50):
        k = int(q * len(y))
        print(f'     상위 {q:>4.0%} 행이 총 손실의 {e[o[:k]].sum() / e.sum() * 100:5.1f}% '
              f'(평균손실 {e[o[:k]].mean():.3f})')

    print('\n  3) 캘리브레이션 — 예측 확신 vs 실제 정답률')
    conf, pred = p.max(1), p.argmax(1)
    hit = (pred == y).astype(float)
    b = np.clip((conf * 10).astype(int), 0, 9)
    ece = 0.0
    for i in range(10):
        m = b == i
        if m.sum() < 20:
            continue
        gap = conf[m].mean() - hit[m].mean()
        ece += m.mean() * abs(gap)
        print(f'     {i / 10:.1f}~{(i + 1) / 10:.1f}  n {int(m.sum()):>5,}  '
              f'확신 {conf[m].mean():.3f}  실제 {hit[m].mean():.3f}  차 {gap:+.3f}'
              + ('  과신' if gap > 0.02 else ('  과소확신' if gap < -0.02 else '')))
    print(f'     ECE {ece:.4f}   (전체 차 {conf.mean() - hit.mean():+.4f})')

    print('\n  4) 축 분해 + 회수율')
    print(f'     tie축  {rep["tie축"]:.4f}  (총 손실의 {rep["tie축"] / tot * 100:4.1f}%)  '
          f'무정보 {NOINFO_TIE:.4f} 대비 {(1 - rep["tie축"] / NOINFO_TIE) * 100:5.2f}% 회수')
    print(f'     승부축 {rep["승부축"]:.4f}  (총 손실의 {rep["승부축"] / tot * 100:4.1f}%)  '
          f'무정보 {NOINFO_WIN:.4f} 대비 {(1 - rep["승부축"] / NOINFO_WIN) * 100:5.2f}% 회수')
    print(f'     오라클: tie 를 완벽히 맞히면 {rep["승부축"]:.4f} 만 남고, '
          f'승부를 완벽히 맞히면 {rep["tie축"]:.4f} 가 남는다')

    print('\n  5) 전문가 tie 판정 합의 층화')
    strata = (('둘 다 tie', ABE_TIE & TB_TIE), ('둘 다 비-tie', ~ABE_TIE & ~TB_TIE),
              ('ABE만 tie', ABE_TIE & ~TB_TIE), ('TB만 tie', ~ABE_TIE & TB_TIE))
    for nm, m in strata:
        if m.sum() < 30:
            print(f'     {nm:<12} n {int(m.sum()):>6,}  (표본 부족)')
            continue
        print(f'     {nm:<12} n {int(m.sum()):>6,} ({m.mean():5.1%})  실제 tie율 {(y[m] == 2).mean():.3f}  '
              f'평균손실 {e[m].mean():.4f}  기여 {e[m].sum() / e.sum() * 100:5.1f}%')

# 최선 구성 + arm 별 3종 앙상블을 해부한다
_best = min(R, key=lambda k: R[k]['mean']['총 ll'])
dissect(R[_best]['oof'], Y, f'최선 단일: {_best[0]}·{_best[1]}')
for arm in ARMS:
    lp = np.mean([np.log(np.clip(R[(arm, n)]['oof'], EPS, 1)) for n in FACTORY], axis=0)
    pe = np.exp(lp)
    pe /= pe.sum(1, keepdims=True)
    dissect(pe, Y, f'{arm} 3종 앙상블')

### 배포용 최종 적합 + 저장

CV 는 측정 전용이었다. 배포 모델은 **stage2 전량**으로 다시 적합한다 —
ES 는 같은 비율(10%)을 한 번 떼어 쓰고, 그 결과 `best_iteration` 이 CV 의 중앙값과 크게 어긋나면
데이터량 변화 이상의 무언가가 있다는 뜻이라 경고를 찍는다.

In [ ]:
FINAL = {}
tr_all, es_all = train_test_split(np.arange(len(Y)), test_size=int(round(INNER_FRAC * len(Y))),
                                  stratify=Y, random_state=SEED, shuffle=True)
for arm, (X, cols) in ARMS.items():
    for name in FACTORY:
        if USE_SWAP:
            Xf = np.vstack([X[tr_all], X[tr_all][:, PERM[arm]]])
            yf = np.hstack([Y[tr_all], swap_y(Y[tr_all])])
        else:
            Xf, yf = X[tr_all], Y[tr_all]
        m = fit_one(name, FACTORY[name](), Xf, yf, X[es_all], Y[es_all])
        FINAL[(arm, name)] = m
        bi, med = best_iter(name, m), int(np.median(R[(arm, name)]['iters']))
        flag = '  경고: CV 중앙값과 2배 이상 차이' if bi and (bi > 2 * med or bi * 2 < med) else ''
        print(f'{arm:<4} {name:<5} best_iter {bi:>5} (CV 중앙값 {med:>5}){flag}')

for (arm, name), m in FINAL.items():
    d = MODEL_DIR / arm
    os.makedirs(d, exist_ok=True)
    if name == 'lgbm':
        m.booster_.save_model(str(d / 'lgbm.txt'))
    elif name == 'xgb':
        m.save_model(str(d / 'xgb.json'))
    else:
        m.save_model(str(d / 'cat.cbm'))

meta = {
    'created': time.strftime('%Y-%m-%d %H:%M'),
    'stage2_rows': int(len(IDS)), 'n_splits': N_SPLITS, 'inner_frac': INNER_FRAC,
    'use_swap': USE_SWAP, 'arith_mode': ARITH_MODE, 'ckpt_select': CKPT_SELECT,
    'experts': {
        'abe': {'step': ABE_STEP, 'T': ABE_T, 'n_class': 3, 'root': str(ABE_ROOT)},
        'tb': {'step': TB_STEP, 'T': TB_T, 'n_class': 2, 'root': str(TB_ROOT),
               'note': 'swap 불변 — [1,0,2] 재정렬 금지. 빈 응답은 배포본 AttnPool 의 '
                       'nan_to_num 가드로 0 벡터 처리'},
        'arith': {'models': list(ARITH_MODELS), 'npz': str(ARITH_NPZ)}},
    'columns': {a: c for a, (X, c) in ARMS.items()},
    'swap_perm': {a: PERM[a].tolist() for a in ARMS},
    'cv_mean': {f'{a}·{n}': {k: float(v) for k, v in R[(a, n)]['mean'].items()} for a, n in R},
    'cv_std_ll': {f'{a}·{n}': float(R[(a, n)]['std']['총 ll']) for a, n in R},
    'best_iters_cv': {f'{a}·{n}': R[(a, n)]['iters'] for a, n in R},
}
with open(MODEL_DIR / 'boost_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

np.savez_compressed(OUT / 'boost_oof.npz', ids=IDS, y=Y,
                    **{f'oof__{a}__{n}': R[(a, n)]['oof'].astype(np.float32) for a, n in R})
print(f'\nsaved: {MODEL_DIR}/{{AT,ATR}}/{{lgbm.txt,xgb.json,cat.cbm}}')
print(f'saved: {MODEL_DIR}/boost_meta.json  |  {OUT}/boost_oof.npz')

## 읽는 법 / 다음 단계

1. **arm 비교가 이 노트북의 본론이다.** `AT -> ATR` 이 문턱 0.0005 를 넘겨 유의하면 산술 전문가가
   앙상블에 실제로 기여하는 것이고, 노이즈면 ABE+TB 만으로 충분하다는 뜻이다 — 그 경우 제출
   파이프라인에서 피쳐 추출·트리 추론을 통째로 빼 시간 예산을 벌 수 있다.
2. **fold 표준편차를 함께 본다.** 평균 차이가 std 보다 작으면 fold 우연으로 뒤집힐 수 있는 차이다.
3. **축 분해**로 어디서 벌었는지 본다. tie 축이 총 손실의 59% 인데 회수율이 낮았다 —
   TB 를 새로 학습한 뒤 tie 축이 얼마나 내려왔는지가 이번 세대의 성적표다.
4. **valid/hold 반쪽 차이**가 크면 1차 모델 선택 편향이 stage2 로 새어든 신호다(§1.6 보조 진단).
5. 배포 시 추론 순서: ABE(2뷰 TTA) → TB(2뷰 TTA) → Arith(피쳐 → 트리 3종 TTA) → 이 부스터.
   **열 순서는 `boost_meta.json` 의 `columns` 가 계약이다.** 순서가 어긋나면 조용히 틀린 답을 낸다.